# {{PROJECT_NAME}}

KFP v2 eval-first fine-tuning pipeline on the Miramar platform.

**Development workflow:**
1. Edit `config.yaml` — set model ID, datasets, thresholds, and judge prompt
2. Edit `formatters.py` — add one formatter function per dataset, register in `FORMATTERS`
3. Write step logic in the `@dsl.component` cells below
4. Save (`Ctrl+S`), run the **Build → `pipeline.py`** cell
5. Trigger **Deploy to KFP** workflow (or run **Compile & Submit** below)

**Pipeline DAG:**
```
prepare_dataset ─┬─► baseline_eval ─────────────────────────────►─┐
                 │                                                  │
                 └─► fine_tune ─┬─► post_finetune_eval ──────────►─┤
                                │                                  │
                                └─► safety_eval ────────────────►─┤
                                                                   │
                                                           deployment_gate
```

## Step Development

Write step logic inside each `@dsl.component` function body.

- **All imports must be inside the function body** — each component runs in its own container
- Set `base_image` and `packages_to_install` to match the step's runtime requirements
- Use `Input[T]` / `Output[T]` to pass artifacts between steps
- GPU steps: call `.set_gpu_limit(1).set_memory_limit("64G")` in the pipeline cell

In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics, Artifact

### prepare_dataset

Loads datasets from HuggingFace, applies formatters from `formatters.py`, splits into train/val/test.
The Build cell inlines `formatters.py` at the `# <<< FORMATTERS_INJECT >>>` marker.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=[
        "datasets<3.0",
        "huggingface_hub>=0.21.2,<0.24",
        "transformers",
    ],
)
def prepare_dataset(
    dataset_names: list,
    val_size: float,
    test_size: float,
    train_out: Output[Dataset],
    val_out: Output[Dataset],
    test_out: Output[Dataset],
):
    import json, pathlib, random

    # <<< FORMATTERS_INJECT >>>
    # formatters.py is inlined here by the Build cell. Do not remove this marker.

    # <<< LOADERS_INJECT >>>
    # loaders.py is inlined here by the Build cell. Do not remove this marker.

    missing = [n for n in dataset_names if n not in LOADERS]
    if missing:
        raise ValueError(f"No loader for: {missing}. Add to loaders.py LOADERS dict.")

    all_rows = []
    for name in dataset_names:
        ds = LOADERS[name]()
        all_rows.extend(
            {"instruction": r["instruction"], "response": r["response"], "source": r["source"]}
            for r in ds
        )

    random.shuffle(all_rows)
    n = len(all_rows)
    n_val  = max(1, int(n * val_size))
    n_test = max(1, int(n * test_size))
    train_rows = all_rows[n_val + n_test:]
    val_rows   = all_rows[:n_val]
    test_rows  = all_rows[n_val:n_val + n_test]

    pathlib.Path(train_out.path).write_text(json.dumps(train_rows))
    pathlib.Path(val_out.path).write_text(json.dumps(val_rows))
    pathlib.Path(test_out.path).write_text(json.dumps(test_rows))
    print(f"Dataset split: {len(train_rows)} train / {len(val_rows)} val / {len(test_rows)} test")

### baseline_eval

Evaluates the base model on the validation set before fine-tuning. Records metrics to MLflow.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:25.03-py3",
    packages_to_install=[
        "transformers>=4.45",
        "accelerate",
        "mlflow",
    ],
)
def baseline_eval(
    val: Input[Dataset],
    base_model_id: str,
    eval_sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    metrics: Output[Metrics],
):
    import json, pathlib, mlflow

    val_data = json.loads(pathlib.Path(val.path).read_text())[:eval_sample_size]

    # TODO: load base model and run inference on val_data
    # TODO: compute accuracy (or your relevant metric) against val_data ground truth
    accuracy = 0.0  # placeholder

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    with mlflow.start_run(run_name=f"{run_id}-baseline"):
        mlflow.log_metric("baseline_accuracy", accuracy)

    pathlib.Path(metrics.path).write_text(json.dumps({"baseline_accuracy": accuracy}))
    print(f"Baseline accuracy: {accuracy:.4f}")

### fine_tune

Fine-tunes the base model using LoRA on the training set.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:25.03-py3",
    packages_to_install=[
        "pyarrow>=21.0.0",
        "peft>=0.14.0",
        "trl>=0.14.0",
        "accelerate>=0.27.0",
        "mlflow",
    ],
)
def fine_tune(
    train: Input[Dataset],
    val: Input[Dataset],
    base_model_id: str,
    learning_rate: float,
    num_epochs: int,
    lora_r: int,
    lora_alpha: int,
    run_id: str,
    mlflow_tracking_uri: str,
    ft_model: Output[Model],
):
    import json, pathlib, mlflow

    train_data = json.loads(pathlib.Path(train.path).read_text())
    val_data   = json.loads(pathlib.Path(val.path).read_text())

    # TODO: implement fine-tuning with LoRA.
    # Suggested pattern:
    #   from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
    #   from peft import LoraConfig, get_peft_model
    #   from trl import SFTTrainer
    #   tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    #   model     = AutoModelForCausalLM.from_pretrained(base_model_id, ...)
    #   lora_cfg  = LoraConfig(r=lora_r, lora_alpha=lora_alpha, ...)
    #   model     = get_peft_model(model, lora_cfg)
    #   trainer   = SFTTrainer(model=model, args=TrainingArguments(...), ...)
    #   trainer.train()
    #   model.save_pretrained(ft_model.path)

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    with mlflow.start_run(run_name=f"{run_id}-finetune"):
        mlflow.log_params({"learning_rate": learning_rate, "num_epochs": num_epochs,
                           "lora_r": lora_r, "lora_alpha": lora_alpha})
    print(f"Fine-tuning complete. Model saved to: {ft_model.path}")

### post_finetune_eval

Evaluates the fine-tuned model on the validation set. Results feed into the deployment gate.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:25.03-py3",
    packages_to_install=[
        "transformers>=4.45",
        "peft>=0.13",
        "accelerate",
        "mlflow",
    ],
)
def post_finetune_eval(
    val: Input[Dataset],
    ft_model: Input[Model],
    base_model_id: str,
    eval_sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    metrics: Output[Metrics],
):
    import json, pathlib, mlflow

    val_data = json.loads(pathlib.Path(val.path).read_text())[:eval_sample_size]

    # TODO: load fine-tuned model (PEFT adapter on top of base_model_id)
    # TODO: run inference on val_data and compute accuracy
    accuracy = 0.0  # placeholder

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    with mlflow.start_run(run_name=f"{run_id}-postft-eval"):
        mlflow.log_metric("postft_accuracy", accuracy)

    pathlib.Path(metrics.path).write_text(json.dumps({"postft_accuracy": accuracy}))
    print(f"Post-FT accuracy: {accuracy:.4f}")

### safety_eval

LLM-as-judge evaluation of the fine-tuned model's outputs. Uses `OPENAI_API_KEY` injected from the
`mlabs-api-keys` K8s secret. Judge model and system prompt are set in `config.yaml`.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:25.03-py3",
    packages_to_install=[
        "transformers>=4.45",
        "peft>=0.13",
        "accelerate",
        "openai",
        "mlflow",
    ],
)
def safety_eval(
    val: Input[Dataset],
    ft_model: Input[Model],
    base_model_id: str,
    judge_model_id: str,
    judge_system_prompt: str,
    sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    metrics: Output[Metrics],
):
    import json, pathlib, mlflow
    from openai import OpenAI

    val_data = json.loads(pathlib.Path(val.path).read_text())[:sample_size]

    # TODO: load fine-tuned model and generate responses for val_data samples.
    # Then score each response with the judge LLM.
    # Suggested pattern:
    #   client = OpenAI()  # uses OPENAI_API_KEY env var (injected from mlabs-api-keys)
    #   for example in val_data:
    #       response = model.generate(example["instruction"])
    #       result = client.chat.completions.create(
    #           model=judge_model_id,
    #           messages=[
    #               {"role": "system", "content": judge_system_prompt},
    #               {"role": "user", "content": f"Response: {response}"},
    #           ],
    #           temperature=0.0,
    #       )
    #       # Parse result.choices[0].message.content (JSON) and accumulate scores
    avg_score = 0.0  # placeholder

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    with mlflow.start_run(run_name=f"{run_id}-safety-eval"):
        mlflow.log_metric("safety_avg_score", avg_score)

    pathlib.Path(metrics.path).write_text(json.dumps({"safety_avg_score": avg_score}))
    print(f"Safety avg score: {avg_score:.4f}")

### deployment_gate

Compares baseline vs post-FT accuracy and safety score against thresholds from `config.yaml`.
Fails the pipeline if either check does not pass. On pass, pushes the adapter to GCS.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
)
def deployment_gate(
    test: Input[Dataset],
    ft_model: Input[Model],
    baseline_metrics: Input[Metrics],
    postft_metrics: Input[Metrics],
    safety_metrics: Input[Metrics],
    accuracy_delta_threshold: float,
    safety_score_threshold: float,
    gcs_bucket: str,
    run_id: str,
):
    import json, pathlib

    baseline = json.loads(pathlib.Path(baseline_metrics.path).read_text())
    postft   = json.loads(pathlib.Path(postft_metrics.path).read_text())
    safety   = json.loads(pathlib.Path(safety_metrics.path).read_text())

    # TODO: update metric keys to match what baseline_eval / post_finetune_eval log
    baseline_acc = baseline.get("baseline_accuracy", 0.0)
    postft_acc   = postft.get("postft_accuracy", 0.0)
    safety_score = safety.get("safety_avg_score", 0.0)

    delta = postft_acc - baseline_acc
    passed = delta >= -accuracy_delta_threshold and safety_score >= safety_score_threshold

    print(f"Accuracy delta : {delta:+.4f}  (threshold ≥ {-accuracy_delta_threshold:.4f})")
    print(f"Safety score   : {safety_score:.4f}  (threshold ≥ {safety_score_threshold:.4f})")
    print(f"Gate           : {'PASS' if passed else 'FAIL'}")

    if not passed:
        raise RuntimeError(
            f"Deployment gate failed — accuracy delta {delta:+.4f}, safety score {safety_score:.4f}"
        )

    # TODO: push ft_model adapter files to GCS
    # from google.cloud import storage
    # client = storage.Client()
    # bucket = client.bucket(gcs_bucket)
    # for p in pathlib.Path(ft_model.path).rglob("*"):
    #     bucket.blob(f"{run_id}/{p.relative_to(ft_model.path)}").upload_from_filename(str(p))
    print(f"Adapter would be pushed to gs://{gcs_bucket}/{run_id}/")

### Pipeline

Wire the steps together. Defaults are read from `config.yaml` at import time.
To override a default when submitting a run, pass it as an argument to `create_run_from_pipeline_package`
or set it in the KFP UI.

In [ ]:
import yaml as _yaml, pathlib as _pathlib

_cfg = _yaml.safe_load(_pathlib.Path("config.yaml").read_text())
_dataset_names = [d["name"] for d in _cfg["datasets"]]

from kfp import kubernetes as k8s_ext


@dsl.pipeline(name="{{PROJECT_NAME}}")
def pipeline(
    base_model_id: str = _cfg["model"]["id"],
    judge_model_id: str = _cfg["judge"]["model"],
    judge_system_prompt: str = _cfg["judge"]["system_prompt"],
    dataset_names: list = _dataset_names,
    learning_rate: float = _cfg["training"]["learning_rate"],
    num_epochs: int = _cfg["training"]["num_epochs"],
    lora_r: int = _cfg["lora"]["r"],
    lora_alpha: int = _cfg["lora"]["alpha"],
    val_size: float = _cfg["training"]["val_size"],
    test_size: float = _cfg["training"]["test_size"],
    eval_sample_size: int = _cfg["eval"]["sample_size"],
    safety_sample_size: int = _cfg["eval"]["safety_sample_size"],
    accuracy_delta_threshold: float = _cfg["eval"]["accuracy_delta_threshold"],
    safety_score_threshold: float = _cfg["eval"]["safety_score_threshold"],
    gcs_bucket: str = _cfg["deployment"]["gcs_bucket"],
    run_id: str = "run-001",
    mlflow_tracking_uri: str = "http://mlflow-tracking.mlflow-system.svc.cluster.local",
):
    prep = prepare_dataset(
        dataset_names=dataset_names,
        val_size=val_size,
        test_size=test_size,
    )

    # baseline_eval and fine_tune both start as soon as prep finishes (parallel)
    base_eval = baseline_eval(
        val=prep.outputs["val_out"],
        base_model_id=base_model_id,
        eval_sample_size=eval_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
    )
    base_eval.set_gpu_limit(1).set_memory_limit("64G")

    ft = fine_tune(
        train=prep.outputs["train_out"],
        val=prep.outputs["val_out"],
        base_model_id=base_model_id,
        learning_rate=learning_rate,
        num_epochs=num_epochs,
        lora_r=lora_r,
        lora_alpha=lora_alpha,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
    )
    ft.set_gpu_limit(1).set_memory_limit("96G")

    post_eval = post_finetune_eval(
        val=prep.outputs["val_out"],
        ft_model=ft.outputs["ft_model"],
        base_model_id=base_model_id,
        eval_sample_size=eval_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
    )
    post_eval.set_gpu_limit(1).set_memory_limit("64G")

    safety = safety_eval(
        val=prep.outputs["val_out"],
        ft_model=ft.outputs["ft_model"],
        base_model_id=base_model_id,
        judge_model_id=judge_model_id,
        judge_system_prompt=judge_system_prompt,
        sample_size=safety_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
    )
    safety.set_gpu_limit(1).set_memory_limit("64G")

    gate = deployment_gate(
        test=prep.outputs["test_out"],
        ft_model=ft.outputs["ft_model"],
        baseline_metrics=base_eval.outputs["metrics"],
        postft_metrics=post_eval.outputs["metrics"],
        safety_metrics=safety.outputs["metrics"],
        accuracy_delta_threshold=accuracy_delta_threshold,
        safety_score_threshold=safety_score_threshold,
        gcs_bucket=gcs_bucket,
        run_id=run_id,
    )

    _SECRET = "mlabs-api-keys"
    _SECRET_KEYS = [
        "OPENAI_API_KEY", "HF_TOKEN", "ANTHROPIC_API_KEY",
        "WANDB_API_KEY", "LANGCHAIN_API_KEY", "NGC_API_KEY", "NVIDIA_API_KEY",
    ]
    _key_map = {k: k for k in _SECRET_KEYS}
    for _task in [prep, base_eval, ft, post_eval, safety, gate]:
        k8s_ext.use_secret_as_env(_task, _SECRET, _key_map)

## Build → `pipeline.py`

Save the notebook first (`Ctrl+S`), then run this cell.

What this does:
1. Reads `formatters.py` and inlines it into the `prepare_dataset` component body
2. Copies all other step cells unchanged
3. Copies the pipeline cell (which reads `config.yaml` at import time)
4. Writes `pipeline.py`

In [ ]:
import json, pathlib, re, textwrap


def build_pipeline(
    notebook_path="notebook.ipynb",
    formatters_path="formatters.py",
):
    nb = json.loads(pathlib.Path(notebook_path).read_text())
    formatters_src = pathlib.Path(formatters_path).read_text().rstrip("\n")

    step_srcs, pipeline_src = [], None
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        tags = cell.get("metadata", {}).get("tags", [])
        src = "".join(cell["source"])
        if "kfp_step" in tags:
            if "# <<< FORMATTERS_INJECT >>>" in src:
                indented = textwrap.indent(formatters_src, "    ")
                src = src.replace("    # <<< FORMATTERS_INJECT >>>", indented)
            step_srcs.append(src)
        elif "kfp_pipeline" in tags:
            pipeline_src = src

    if not step_srcs:
        raise RuntimeError("No cells tagged 'kfp_step' found.")
    if pipeline_src is None:
        raise RuntimeError("No cell tagged 'kfp_pipeline' found.")

    names = [
        m.group(1)
        for src in step_srcs
        for m in [re.search(r"^def (\w+)\(", src, re.MULTILINE)]
        if m
    ]

    out  = "# Generated by notebook.ipynb \u2014 do not edit manually.\n"
    out += "# Re-run the Build cell to regenerate.\n\n"
    out += "from kfp import dsl\n"
    out += "from kfp.dsl import Input, Output, Dataset, Model, Metrics, Artifact\n\n\n"
    out += "\n\n\n".join(step_srcs)
    out += "\n\n\n"
    out += pipeline_src
    out += "\n"
    pathlib.Path("pipeline.py").write_text(out)
    print(f"Wrote pipeline.py \u2014 {len(names)} component(s): {', '.join(names)}")


build_pipeline()

## Compile & Submit

Run these cells to compile and submit directly from Jupyter (requires SSH tunnel).

In [ ]:
from kfp import compiler
from pipeline import pipeline

compiler.Compiler().compile(pipeline_func=pipeline, package_path="/tmp/pipeline.yaml")
print("Compiled: /tmp/pipeline.yaml")

In [ ]:
# Requires SSH tunnel: ssh -L 8080:localhost:8080 <user>@spark-79b7.local
import kfp

client = kfp.Client(host="http://localhost:8080")

In [ ]:
run = client.create_run_from_pipeline_package(
    pipeline_file="/tmp/pipeline.yaml",
    arguments={},
    run_name="notebook-run",
)
print(f"Run ID: {run.run_id}")

In [ ]:
import time

run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.state
    print(f"{state}")
    if state in ("SUCCEEDED", "FAILED", "CANCELED", "SKIPPED"):
        break
    time.sleep(30)